In [ ]:
# Author: Niko Bleidistel
# last change: 2026-08-19

# Start

## Package Import

In [ ]:
from pathlib import Path
from os import makedirs
import sys

import pandas as pd
import numpy as np

import importlib

In [ ]:
PYTHON_HELPER_FOLDER = Path(r"py-helpers")

# Add the path to the custom packages to sys.path so that they can be imported
sys.path.append(str(PYTHON_HELPER_FOLDER.resolve()))

# import custom packages
import time_logging as tl
import comsol_data_import as cdi
import advanced_plotting_functions as apf

import comsol_data_plotting2 as cdp
import theory_fit_models as tfm

# reload custom packages (for each execution) to reflect recent changes
_ = importlib.reload(tl)
_ = importlib.reload(cdi)
_ = importlib.reload(apf)
_ = importlib.reload(cdp)
_ = importlib.reload(tfm)

## PATHS

In [ ]:
MAIN_FOLDER_1 = Path(r"R:\Bleidistel_Niko\S\data\high res (try 2)")


INPUT_FOLDER_1 = MAIN_FOLDER_1 / "RESULTS"
OUTPUT_FOLDER_1 = MAIN_FOLDER_1 / "SPECIFIC PLOTS"
SWEEP_OUTPUT_FOLDER_1 = OUTPUT_FOLDER_1 / "SWEEP PLOTS"

makedirs(OUTPUT_FOLDER_1, exist_ok=True)  # create output folder if it doesn't exist

In [ ]:
MAIN_FOLDER_2 = Path(r"R:\Bleidistel_Niko\S\data\high res (try 3)")


INPUT_FOLDER_2 = MAIN_FOLDER_2 / "RESULTS"
OUTPUT_FOLDER_2 = MAIN_FOLDER_2 / "SPECIFIC PLOTS"
SWEEP_OUTPUT_FOLDER_2 = OUTPUT_FOLDER_2 / "SWEEP PLOTS"

makedirs(OUTPUT_FOLDER_2, exist_ok=True)  # create output folder if it doesn't exist

In [ ]:
# initialize time logging
_ = tl.initialize_time_log(OUTPUT_FOLDER_1 / 'time_log.csv')

In [ ]:
# input group structure
FOLDER_01_00 = "01_00-Meshing and POC"
FOLDER_01_XX = "01_xx-Coils and Grid Designs"
FOLDER_02_00 = "02_00-H Designs"

# additional output group structure
FOLDER_01_01 = "01_01-Coils"
FOLDER_01_02 = "01_02-Grid"
FOLDER_01_03 = "01_03-Combined Coils and Grid"

In [ ]:
# Group models which should be compared in the same plot
MODELS_01_00 = [
    "01_00_a-auto mesh",
    "01_00_b-high resolution cuboid",
    "01_00_c-high resolution planes",
    "01_00_d-high resolution planes and increasing areas"
]
MODELS_01_01 = [
    "01_01_a-Round spiral",
    "01_01_b-Rectangular spiral",
]
MODELS_01_02 = [
    "01_02_a-Grid",
]
MODELS_01_03 = [
    "01_03_a-Round spiral combined with grid",
    "01_03_b-Rectangular spiral combined with grid",
]
MODELS_02_00 = [
    # "02_00_a-H design with minimized insulator",
    "02_00_b-H design with full insulator",
]

In [ ]:
TERMINAL_END = "-terminals.csv"
PARAMETER_END = "-parameters.csv"
DEPTH_END = "-depth_exported_data.txt"
HOMOGENEITY_END = "-homogeneity_exported_data.txt"
LONGITUDINAL_END = "-longitudinal_exported_data.txt"
XY_PLANE_END = "-xy_exported_data.txt"
CONDUCTOR_PLANE_END = "-conductor_exported_data.txt"

In [ ]:
DATA_EXPORT = "Data Export"
SWEEP_EXPORT = "Sweep Export"

In [ ]:
SWEEP_01_01 = {
    "01_01_a-Round spiral": "Sweep - N_spiral_turns",
}

SWEEP_01_02 = {
    "01_02_a-Grid": "Sweep - Voltage angles and left_out_lines",
}
LEFT_OUT_LINES = "left_out_lines - " #+ Number

SWEEP_01_03 = {
    # "01_03_a-Round spiral combined with grid": "Sweep - Voltage angles",
    "01_03_b-Rectangular spiral combined with grid": "Sweep - Voltage angles",
}
SWEEP_PARAMETER_DICT_01_03: dict[str, list[float]] = {
        r"$\theta$ $[^\circ]$": [0, 45, 67, 90]
    }

SWEEP_02_00 = {
    # "02_00_a-H design with minimized insulator": "Sweep",
    "02_00_b-H design with full insulator": "Sweep",
}

In [ ]:
if False:
    input_folder = INPUT_FOLDER_1

    def mask(filename):
        return not (str(filename).endswith('.mph') or str(filename).endswith('.csv'))

    import seedir as sd
    sd.seedir(input_folder, style='lines', mask=mask)

In [ ]:
if False:
    folders = [f for f in INPUT_FOLDER_1.rglob("*") if f.is_dir()]
    data_export_folders = [f for f in folders if "Data Export" in f.name]
    sweep_export_folders = [f for f in folders if "Sweep Export" in f.name]
    display(data_export_folders)
    display(sweep_export_folders)

## TRANSLATION (Constants)

In [ ]:
GROUPNAME_01_00 = "Different Meshing Approaches and Proof of Concept"
GROUPNAME_02_00 = r"\enquote{H}-Designs"

GROUPNAME_01_01 = "Coil Designs"
GROUPNAME_01_02 = "Grid Design"
GROUPNAME_01_03 = "Combined Grid and Coil Designs"

In [ ]:
SWEEPNAME_02_00 = r"Current Sweep on \enquote{H}-Designs"

SWEEPNAME_01_01 = r"\enquote{Number of turns} Sweep on Round Coil "
SWEEPNAME_01_02_LOL = r"Number of Lines Sweep on Grid Design"
SWEEPNAME_01_02 = r"Angle Error Analysis on Grid Design"
SWEEPNAME_01_03 = r"Combined Grid and Coil Design"

In [ ]:
# translate comsol label to nicer plot labels
TRANSLATE_PLOTLABELS = {
    # axes
    "x":                r"$x$ $[\mathrm{m}]$",
    "y":                r"$y$ $[\mathrm{m}]$",
    "z":                r"$z$ $[\mathrm{m}]$",

    # parameters
    "mf.normB (T)":     "Magnetic flux density, norm\n\n"+r"$|\vec{B}|$ $[\mathrm{T}]$",
    "mf.Bx (T)":        "Magnetic flux density, $x$-component\n\n"+r"$B_x$ $[\mathrm{T}]$", 
    "mf.By (T)":        "Magnetic flux density, $y$-component\n\n"+r"$B_y$ $[\mathrm{T}]$", 
    "mf.Bz (T)":        "Magnetic flux density, $z$-component\n\n"+r"$B_z$ $[\mathrm{T}]$",

    "T (K)":            "Relative Temperature\n\n"+r"$T$ $[\mathrm{K}]$",
    "V (V)":            "Electric potential\n\n"+r"$V$ $[\mathrm{V}]$",

    "ec.normJ (A/m^2)": "Current density, norm\n\n"+r"$|\vec{J}|$ $[\mathrm{A}/\mathrm{m}^2]$",
    "ec.Jx (A/m^2)":    "Current density, $x$-component\n\n"+r"$J_x$ $[\mathrm{A}/\mathrm{m}^2]$",
    "ec.Jy (A/m^2)":    "Current density, $y$-component\n\n"+r"$J_y$ $[\mathrm{A}/\mathrm{m}^2]$",
    "ec.Jz (A/m^2)":    "Current density, $z$-component\n\n"+r"$J_z$ $[\mathrm{A}/\mathrm{m}^2]$",

    "Set angle (°)":    "Set angle "+r"$\theta$ $[^\circ]$",
    "Angle Error (°)":  "Angle error "+r"$\Delta\theta$ $[^\circ]$",

    # modelnames
    "auto mesh":                                    "automatic",
    "high resolution cuboid":                       "areas",
    "high resolution planes":                       "planes",
    "high resolution planes and increasing areas":  "planes and areas",

    "Round spiral":                                 "Round coil",
    "Rectangular spiral":                           "Rectangular coil",

    "Grid":                                         "Grid",
    
    "Round spiral combined with grid":              "Round coil and grid",
    "Rectangular spiral combined with grid":        "Rectangular coil and grid",

    "H design with minimized insulator":            "Minimized insulator",
    "H design with full insulator":                 "Full insulator",

    # parameters
    "I_conductor_terminal":             r"$I$ $[\mathrm{A}]$",
    "I_spiral_terminal":                r"$I_{\mathrm{coil}}$ $[\mathrm{A}]$",
    
    "I_conductor_A_terminal":           r"$I_{\uparrow}$ $[\mathrm{A}]$",
    "I_conductor_B_minus_terminal":     r"$I_{\leftarrow}$ $[\mathrm{A}]$",
    "I_conductor_B_plus_terminal":      r"$I_{\rightarrow}$ $[\mathrm{A}]$",

    "N_spiral_turns":                   r"$N_{\mathrm{turns}}$",
    "N_grid":                           r"$N_{\mathrm{grid}}$",

    "V_Grid":                           r"$V_{\mathrm{pp}}$ $[\mathrm{V}]$",
}

In [ ]:
# X_AXIS_PARAMS = ["x", "y", "z"]
# Y_AXIS_PARAMS = [key for key in TRANSLATE_PLOTLABELS.keys() if key not in X_AXIS_PARAMS]

In [ ]:
Y_AXIS_PARAMS = [
    # "mf.normB (T)",
    "mf.Bx (T)",
    "mf.By (T)",
    "mf.Bz (T)",
    # "T (K)", # added where it is needed
]
CON_PLANE_PARAMS = [
    "ec.normJ (A/m^2)",
    # "ec.Jx (A/m^2)",
    # "ec.Jy (A/m^2)",
    # "ec.Jz (A/m^2)",
    "V (V)",
]

In [ ]:
Z_VALUES = [
    -3e-06,
    -6e-06, 
    -9e-06,
]

# SIMULATIONS

# 01_00-Meshing and POC

## z-axis (depth)

In [ ]:
cdp.zaxis_plot(
    translation_dict = TRANSLATE_PLOTLABELS,

    # theory
    magnetic_theory = cdp.add_magnetic_theory_01_00,
    temperature_theory = cdp.add_temperature_theory_01_00,

    #paths
    output_folder = OUTPUT_FOLDER_1,
    input_folder = INPUT_FOLDER_1,
    modelfolder = FOLDER_01_00,
    groupname = GROUPNAME_01_00,
    modelnames = MODELS_01_00,
    
    ending = DEPTH_END,
    data_export = DATA_EXPORT,

    #plotting
    fraction = 1.0,
    y_axis_params=Y_AXIS_PARAMS + ["T (K)"],

    title_parameters = ["I_conductor_terminal"],
    title_param_dict = None,
    
    label_parameters = None,
    label_param_dict = None,
)

## y-axis (homogenity)

In [ ]:
cdp.yaxis_plot(
    translation_dict = TRANSLATE_PLOTLABELS,

    # theory
    magnetic_theory = cdp.add_magnetic_theory_01_00,
    temperature_theory = cdp.add_temperature_theory_01_00,

    #paths
    output_folder = OUTPUT_FOLDER_1,
    input_folder = INPUT_FOLDER_1,
    modelfolder = FOLDER_01_00,
    groupname = GROUPNAME_01_00,
    modelnames = MODELS_01_00,
    
    ending = HOMOGENEITY_END,
    data_export = DATA_EXPORT,

    #plotting
    fraction = 1.0,
    y_axis_params=Y_AXIS_PARAMS + ["T (K)"],

    plot_z = Z_VALUES[1:-1],
    insulator = True,

    width_limit_param = "conductor_all_width",
    width_limit = None,

    title_parameters = ["I_conductor_terminal"],
    title_param_dict = None,

    label_parameters = None,
    label_param_dict = None,
)

## x-axis (longitudinal)

In [ ]:
cdp.xaxis_plot(
    translation_dict = TRANSLATE_PLOTLABELS,

    # theory
    magnetic_theory = cdp.add_magnetic_theory_01_00,
    temperature_theory = None,

    #paths
    output_folder = OUTPUT_FOLDER_1,
    input_folder = INPUT_FOLDER_1,
    modelfolder = FOLDER_01_00,
    groupname = GROUPNAME_01_00,
    modelnames = MODELS_01_00,
    
    ending = LONGITUDINAL_END,
    data_export = DATA_EXPORT,

    #plotting
    fraction = 1.0,
    y_axis_params=Y_AXIS_PARAMS,

    plot_z = Z_VALUES[1:-1],
    insulator = True,

    length_limit_param = "conductor_all_length",
    length_limit = None,

    title_parameters = ["I_conductor_terminal"],
    title_param_dict = None,

    label_parameters = None,
    label_param_dict = None,
)

## xy-plane

In [ ]:
cdp.xyplane_plot(
    translation_dict = TRANSLATE_PLOTLABELS,

    #paths
    output_folder = OUTPUT_FOLDER_1,
    input_folder = INPUT_FOLDER_1,
    modelfolder = FOLDER_01_00,
    groupname = GROUPNAME_01_00,
    modelnames = MODELS_01_00,
    
    ending = XY_PLANE_END,
    data_export = DATA_EXPORT,

    #plotting
    fraction = 1.0,
    z_axis_params= ["T (K)"], # Y_AXIS_PARAMS + ["T (K)"],

    title_parameters = ["I_conductor_terminal"],
    title_param_dict = None,

    length_limit_param = "conductor_all_length",
    length_limit = None,

    width_limit_param = "conductor_all_width",
    width_limit = None,
)

# 01_01-Coil Designs

## z-axis (depth)

In [ ]:
# cdp.zaxis_plot(
#     translation_dict = TRANSLATE_PLOTLABELS,

#     # theory
#     magnetic_theory = None,
#     temperature_theory = None,

#     #paths
#     output_folder = OUTPUT_FOLDER,
#     input_folder = INPUT_FOLDER,
#     modelfolder = FOLDER_01_XX,
#     groupname = GROUPNAME_01_01,
#     modelnames = MODELS_01_01,
    
#     ending = DEPTH_END,
#     data_export = DATA_EXPORT,

#     #plotting
#     fraction = 1.0,
#     y_axis_params=Y_AXIS_PARAMS,

#     title_parameters = ["I_spiral_terminal"],
#     title_param_dict = None,

#     label_parameters = None,
#     label_param_dict = None,
# )

## y-axis (homogenity)

In [ ]:
cdp.yaxis_plot(
    translation_dict = TRANSLATE_PLOTLABELS,

    # theory
    magnetic_theory = None,
    temperature_theory = None,

    #paths
    output_folder = OUTPUT_FOLDER_1,
    input_folder = INPUT_FOLDER_1,
    modelfolder = FOLDER_01_XX,
    groupname = GROUPNAME_01_01,
    modelnames = MODELS_01_01,
    
    ending = HOMOGENEITY_END,
    data_export = DATA_EXPORT,

    #plotting
    fraction = 1.0,
    y_axis_params=Y_AXIS_PARAMS,

    plot_z = Z_VALUES[1:],
    insulator = False,

    width_limit_param = None,
    width_limit = None,

    title_parameters = ["I_spiral_terminal"],
    title_param_dict = None,

    label_parameters = None,
    label_param_dict = None,
)

## x-axis (longitudinal)

In [ ]:
cdp.xaxis_plot(
    translation_dict = TRANSLATE_PLOTLABELS,

    # theory
    magnetic_theory = None,
    temperature_theory = None,

    #paths
    output_folder = OUTPUT_FOLDER_1,
    input_folder = INPUT_FOLDER_1,
    modelfolder = FOLDER_01_XX,
    groupname = GROUPNAME_01_01,
    modelnames = MODELS_01_01,
    
    ending = LONGITUDINAL_END,
    data_export = DATA_EXPORT,

    #plotting
    fraction = 1.0,
    y_axis_params=Y_AXIS_PARAMS,

    plot_z = Z_VALUES[1:],
    insulator = False,
    
    length_limit_param = None,
    length_limit = None,

    title_parameters = ["I_spiral_terminal"],
    title_param_dict = None,

    label_parameters = None,
    label_param_dict = None,
)

## xy-plane

In [ ]:
cdp.xyplane_plot(
    translation_dict = TRANSLATE_PLOTLABELS,

    #paths
    output_folder = OUTPUT_FOLDER_1,
    input_folder = INPUT_FOLDER_1,
    modelfolder = FOLDER_01_XX,
    groupname = GROUPNAME_01_01,
    modelnames = MODELS_01_01,
    
    ending = XY_PLANE_END,
    data_export = DATA_EXPORT,

    #plotting
    fraction = 1.0,
    z_axis_params=Y_AXIS_PARAMS,

    title_parameters = ["I_spiral_terminal"],
    title_param_dict = None,

    length_limit_param = "conductor_all_length",
    length_limit = None,

    width_limit_param = "conductor_all_length",
    width_limit = None,
)

# 01_02-Grid Design

## z-axis (depth)

In [ ]:
# cdp.zaxis_plot(
#     translation_dict = TRANSLATE_PLOTLABELS,

#     # theory
#     magnetic_theory = None,
#     temperature_theory = None,

#     #paths
#     output_folder = OUTPUT_FOLDER,
#     input_folder = INPUT_FOLDER,
#     modelfolder = FOLDER_01_XX,
#     groupname = SWEEPNAME_01_02_LOL,
#     modelnames = MODELS_01_02,
    
#     ending = DEPTH_END,
#     data_export = SWEEP_EXPORT,

#     #plotting
#     fraction = 1.0,
#     y_axis_params=Y_AXIS_PARAMS,
#     title_parameters = ["V_Grid"],

#     # alibi sweep
#     sweep_dict = SWEEP_01_02,
#     label_parameters = ["N_grid"],
#     label_param_dict={"left_out_lines": [0, 1, 2]},
#     only_iterations = [15],
#     display_sweep_param_dict= False,
# )

## y-axis (homogenity)

In [ ]:
cdp.yaxis_plot(
    translation_dict = TRANSLATE_PLOTLABELS,

    # theory
    magnetic_theory = None,
    temperature_theory = None,

    #paths
    output_folder = OUTPUT_FOLDER_1,
    input_folder = INPUT_FOLDER_1,
    modelfolder = FOLDER_01_XX,
    groupname = GROUPNAME_01_02,
    modelnames = MODELS_01_02,
    
    ending = HOMOGENEITY_END,
    data_export = DATA_EXPORT,

    #plotting
    fraction = 1.0,
    y_axis_params=Y_AXIS_PARAMS,

    plot_z = Z_VALUES,
    insulator = False,

    width_limit_param = None,
    width_limit = None,

    title_parameters = ["V_Grid"],
    title_param_dict = {r"$\theta$ $[^\circ]$": 45}, 

    label_parameters = None,
    label_param_dict = None,
)

## x-axis (longitudinal)

In [ ]:
cdp.xaxis_plot(
    translation_dict = TRANSLATE_PLOTLABELS,

    # theory
    magnetic_theory = None,
    temperature_theory = None,

    #paths
    output_folder = OUTPUT_FOLDER_1,
    input_folder = INPUT_FOLDER_1,
    modelfolder = FOLDER_01_XX,
    groupname = GROUPNAME_01_02,
    modelnames = MODELS_01_02,
    
    ending = LONGITUDINAL_END,
    data_export = DATA_EXPORT,

    #plotting
    fraction = 1.0,
    y_axis_params=Y_AXIS_PARAMS,

    plot_z = Z_VALUES,
    insulator = False,

    length_limit_param = None,
    length_limit = None,

    title_parameters = ["V_Grid"],
    title_param_dict = {r"$\theta$ $[^\circ]$": 45}, 

    label_parameters = None,
    label_param_dict = None,
)

## xy-plane

In [ ]:
cdp.xyplane_plot(
    translation_dict = TRANSLATE_PLOTLABELS,

    #paths
    output_folder = OUTPUT_FOLDER_1,
    input_folder = INPUT_FOLDER_1,
    modelfolder = FOLDER_01_XX,
    groupname = GROUPNAME_01_02,
    modelnames = MODELS_01_02,
    
    ending = XY_PLANE_END,
    data_export = DATA_EXPORT,

    #plotting
    fraction = 1.0,
    z_axis_params=Y_AXIS_PARAMS + ["T (K)"],

    title_parameters = ["V_Grid"],
    title_param_dict = {r"$\theta$ $[^\circ]$": 45}, 

    length_limit_param = "conductor_all_length",
    length_limit = None,

    width_limit_param = "conductor_all_length",
    width_limit = None,
)

## conductor-plane

In [ ]:
cdp.xyplane_plot(
    translation_dict = TRANSLATE_PLOTLABELS,

    #paths
    output_folder = OUTPUT_FOLDER_1,
    input_folder = INPUT_FOLDER_1,
    modelfolder = FOLDER_01_XX,
    groupname = GROUPNAME_01_02,
    modelnames = MODELS_01_02,
    
    ending = CONDUCTOR_PLANE_END,
    data_export = DATA_EXPORT,

    #plotting
    fraction = 1.0,
    z_axis_params=CON_PLANE_PARAMS,

    title_parameters = ["V_Grid"],
    title_param_dict = {r"$\theta$ $[^\circ]$": 45}, 

    length_limit_param = "conductor_all_length",
    length_limit = None,

    width_limit_param = "conductor_all_length",
    width_limit = None,
)

# 01_03-Combined Grid and Coil Designs

## z-axis (depth)

In [ ]:
# cdp.zaxis_plot(
#     translation_dict = TRANSLATE_PLOTLABELS,

#     # theory
#     magnetic_theory = None,
#     temperature_theory = None,

#     #paths
#     output_folder = OUTPUT_FOLDER_1,
#     input_folder = INPUT_FOLDER_1,
#     modelfolder = FOLDER_01_XX,
#     groupname = GROUPNAME_01_03,
#     modelnames = MODELS_01_03,
    
#     ending = DEPTH_END,
#     data_export = DATA_EXPORT,

#     #plotting
#     fraction = 1.0,
#     y_axis_params=Y_AXIS_PARAMS,

#     title_parameters = ["V_Grid", "I_spiral_terminal"],
#     title_param_dict = {r"$\theta$ $[^\circ]$": 45}, 

#     label_parameters = None,
#     label_param_dict = None,
# )

## y-axis (homogenity)

In [ ]:
cdp.yaxis_plot(
    translation_dict = TRANSLATE_PLOTLABELS,

    # theory
    magnetic_theory = None,
    temperature_theory = None,

    #paths
    output_folder = OUTPUT_FOLDER_1,
    input_folder = INPUT_FOLDER_1,
    modelfolder = FOLDER_01_XX,
    groupname = GROUPNAME_01_03,
    modelnames = MODELS_01_03,
    
    ending = HOMOGENEITY_END,
    data_export = DATA_EXPORT,

    #plotting
    fraction = 1.0,
    y_axis_params=Y_AXIS_PARAMS,

    plot_z = Z_VALUES[1:],
    insulator = False,

    width_limit_param = None,
    width_limit = None,

    title_parameters = ["V_Grid", "I_spiral_terminal"],
    title_param_dict = {r"$\theta$ $[^\circ]$": 45}, 

    label_parameters = None,
    label_param_dict = None,
)

## x-axis (longitudinal)

In [ ]:
cdp.xaxis_plot(
    translation_dict = TRANSLATE_PLOTLABELS,

    # theory
    magnetic_theory = None,
    temperature_theory = None,

    #paths
    output_folder = OUTPUT_FOLDER_1,
    input_folder = INPUT_FOLDER_1,
    modelfolder = FOLDER_01_XX,
    groupname = GROUPNAME_01_03,
    modelnames = MODELS_01_03,
    
    ending = LONGITUDINAL_END,
    data_export = DATA_EXPORT,

    #plotting
    fraction = 1.0,
    y_axis_params=Y_AXIS_PARAMS,

    plot_z = Z_VALUES[1:],
    insulator = False,
    
    length_limit_param = None,
    length_limit = None,

    title_parameters = ["V_Grid", "I_spiral_terminal"],
    title_param_dict = {r"$\theta$ $[^\circ]$": 45}, 

    label_parameters = None,
    label_param_dict = None,
)

## xy-plane

In [ ]:
cdp.xyplane_plot(
    translation_dict = TRANSLATE_PLOTLABELS,

    #paths
    output_folder = OUTPUT_FOLDER_1,
    input_folder = INPUT_FOLDER_1,
    modelfolder = FOLDER_01_XX,
    groupname = GROUPNAME_01_03,
    modelnames = MODELS_01_03,
    
    ending = XY_PLANE_END,
    data_export = DATA_EXPORT,

    #plotting
    fraction = 1.0,
    z_axis_params=Y_AXIS_PARAMS,

    title_parameters = ["V_Grid", "I_spiral_terminal"],
    title_param_dict = {r"$\theta$ $[^\circ]$": 45}, 

    length_limit_param = "conductor_all_length",
    length_limit = None,

    width_limit_param = "conductor_all_length",
    width_limit = None,
)

## conductor-plane

In [ ]:
cdp.xyplane_plot(
    translation_dict = TRANSLATE_PLOTLABELS,

    #paths
    output_folder = OUTPUT_FOLDER_1,
    input_folder = INPUT_FOLDER_1,
    modelfolder = FOLDER_01_XX,
    groupname = GROUPNAME_01_03,
    modelnames = MODELS_01_03,
    
    ending = CONDUCTOR_PLANE_END,
    data_export = DATA_EXPORT,

    #plotting
    fraction = 1.0,
    z_axis_params=CON_PLANE_PARAMS,

    title_parameters = ["V_Grid", "I_spiral_terminal"],
    title_param_dict = {r"$\theta$ $[^\circ]$": 45}, 

    length_limit_param = "conductor_all_length",
    length_limit = None,

    width_limit_param = "conductor_all_length",
    width_limit = None,
)

# 02_00-H Designs

## z-axis (depth)

In [ ]:
cdp.zaxis_plot(
    translation_dict = TRANSLATE_PLOTLABELS,

    # theory
    magnetic_theory = cdp.add_magnetic_theory_02_00,
    temperature_theory = None,

    #paths
    output_folder = OUTPUT_FOLDER_1,
    input_folder = INPUT_FOLDER_1,
    modelfolder = FOLDER_02_00,
    groupname = GROUPNAME_02_00,
    modelnames = MODELS_02_00,
    
    ending = DEPTH_END,
    data_export = DATA_EXPORT,

    #plotting
    fraction = 1.0,
    y_axis_params=Y_AXIS_PARAMS,

    title_parameters = [
        "I_conductor_A_terminal", 
        "I_conductor_B_minus_terminal", 
        "I_conductor_B_plus_terminal",
        ],
    title_param_dict = None,
    
    label_parameters = None,
    label_param_dict = None,
)

## y-axis (homogenity)

In [ ]:
cdp.yaxis_plot(
    translation_dict = TRANSLATE_PLOTLABELS,

    # theory
    magnetic_theory = cdp.add_magnetic_theory_02_00,
    temperature_theory = None,

    #paths
    output_folder = OUTPUT_FOLDER_1,
    input_folder = INPUT_FOLDER_1,
    modelfolder = FOLDER_02_00,
    groupname = GROUPNAME_02_00,
    modelnames = MODELS_02_00,
    
    ending = HOMOGENEITY_END,
    data_export = DATA_EXPORT,

    #plotting
    fraction = 1.0,
    y_axis_params=Y_AXIS_PARAMS,

    plot_z = Z_VALUES,
    insulator = True,

    width_limit_param = None, # special case (hardcoded in function)
    width_limit = None,    

    title_parameters = [
        "I_conductor_A_terminal", 
        "I_conductor_B_minus_terminal", 
        "I_conductor_B_plus_terminal",
        ],
    title_param_dict = None,
    
    label_parameters = None,
    label_param_dict = None,
)

## x-axis (longitudinal)

In [ ]:
cdp.xaxis_plot(
    translation_dict = TRANSLATE_PLOTLABELS,

    # theory
    magnetic_theory = cdp.add_magnetic_theory_02_00,
    temperature_theory = None,

    #paths
    output_folder = OUTPUT_FOLDER_1,
    input_folder = INPUT_FOLDER_1,
    modelfolder = FOLDER_02_00,
    groupname = GROUPNAME_02_00,
    modelnames = MODELS_02_00,
    
    ending = LONGITUDINAL_END,
    data_export = DATA_EXPORT,

    #plotting
    fraction = 1.0,
    y_axis_params=Y_AXIS_PARAMS,

    plot_z = Z_VALUES,
    insulator = True,

    length_limit_param = None,
    length_limit = None,

    title_parameters = [
        "I_conductor_A_terminal", 
        "I_conductor_B_minus_terminal", 
        "I_conductor_B_plus_terminal",
        ],
    title_param_dict = None,
    
    label_parameters = None,
    label_param_dict = None,
)

## xy-plane

In [ ]:
# cdp.xyplane_plot(
#     translation_dict = TRANSLATE_PLOTLABELS,

#     #paths
#     output_folder = OUTPUT_FOLDER,
#     input_folder = INPUT_FOLDER,
#     modelfolder = FOLDER_02_00,
#     groupname = GROUPNAME_02_00,
#     modelnames = MODELS_02_00,
    
#     ending = XY_PLANE_END,
#     data_export = DATA_EXPORT,

#     #plotting
#     fraction = 1.0,
#     z_axis_params=Y_AXIS_PARAMS,

#     length_limit_param = "conductor_B_length",
#     length_limit = None,

#     width_limit_param = None, # special case (hardcoded in function)
#     width_limit = None,

#     title_parameters = [
#         "I_conductor_A_terminal", 
#         "I_conductor_B_minus_terminal", 
#         "I_conductor_B_plus_terminal",
#         ],
#     title_param_dict = None,
# )

# SWEEPS

# 02_00-Current Sweeps

## z-axis (depth)

In [ ]:
cdp.zaxis_plot(
    translation_dict = TRANSLATE_PLOTLABELS,

    # theory
    magnetic_theory = None,
    temperature_theory = None,

    #paths
    output_folder = SWEEP_OUTPUT_FOLDER_2,
    input_folder = INPUT_FOLDER_2,
    modelfolder = FOLDER_02_00,
    groupname = SWEEPNAME_02_00,
    modelnames = list(SWEEP_02_00.keys()),
    
    ending = DEPTH_END,
    data_export = SWEEP_EXPORT,

    #plotting
    fraction = 1.0,
    y_axis_params=Y_AXIS_PARAMS,

    title_parameters = None,
    title_param_dict = None,
    
    label_parameters = [
        "I_conductor_A_terminal", 
        "I_conductor_B_minus_terminal", 
        "I_conductor_B_plus_terminal",
        ],
    label_param_dict = None,

    # sweep
    sweep_dict = SWEEP_02_00,
    only_iterations = None
)

## y-axis (homogenity)

In [ ]:
cdp.yaxis_plot(
    translation_dict = TRANSLATE_PLOTLABELS,

    # theory
    magnetic_theory = None,
    temperature_theory = None,

    #paths
    output_folder = SWEEP_OUTPUT_FOLDER_2,
    input_folder = INPUT_FOLDER_2,
    modelfolder = FOLDER_02_00,
    groupname = SWEEPNAME_02_00,
    modelnames = list(SWEEP_02_00.keys()),
    
    ending = HOMOGENEITY_END,
    data_export = SWEEP_EXPORT,

    #plotting
    fraction = 1.0,
    y_axis_params=Y_AXIS_PARAMS,

    plot_z = Z_VALUES[1:-1],
    insulator = True,

    width_limit_param = None, # special case (hardcoded in function)
    width_limit = None,

    title_parameters = None,
    title_param_dict = None,
    
    label_parameters = [
        "I_conductor_A_terminal", 
        "I_conductor_B_minus_terminal", 
        "I_conductor_B_plus_terminal",
        ],
    label_param_dict = None,

    #sweep
    sweep_dict = SWEEP_02_00,
    only_iterations = None
)

## x-axis (longitudinal)

In [ ]:
cdp.xaxis_plot(
    translation_dict = TRANSLATE_PLOTLABELS,

    # theory
    magnetic_theory = None,
    temperature_theory = None,

    #paths
    output_folder = SWEEP_OUTPUT_FOLDER_2,
    input_folder = INPUT_FOLDER_2,
    modelfolder = FOLDER_02_00,
    groupname = SWEEPNAME_02_00,
    modelnames = list(SWEEP_02_00.keys()),
    
    ending = LONGITUDINAL_END,
    data_export = SWEEP_EXPORT,

    #plotting
    fraction = 1.0,
    y_axis_params=Y_AXIS_PARAMS,

    plot_z = Z_VALUES[1:-1],
    insulator = True,

    length_limit_param = None,
    length_limit = None,

    title_parameters = None,
    title_param_dict = None,
    
    label_parameters = [
        "I_conductor_A_terminal", 
        "I_conductor_B_minus_terminal", 
        "I_conductor_B_plus_terminal",
        ],
    label_param_dict = None,

    #sweep
    sweep_dict = SWEEP_02_00,
    only_iterations = None
)

# 01_01-Round coil N_spiral_turns Sweep

## z-axis (depth)

In [ ]:
# cdp.zaxis_plot(
#     translation_dict = TRANSLATE_PLOTLABELS,

#     # theory
#     magnetic_theory = None,
#     temperature_theory = None,

#     #paths
#     output_folder = SWEEP_OUTPUT_FOLDER,
#     input_folder = INPUT_FOLDER,
#     modelfolder = FOLDER_01_XX,
#     groupname = SWEEPNAME_01_01,
#     modelnames = list(SWEEP_01_01.keys()),
    
#     ending = DEPTH_END,
#     data_export = SWEEP_EXPORT,

#     #plotting
#     fraction = 1.0,
#     y_axis_params=Y_AXIS_PARAMS,

#     title_parameters = ["I_spiral_terminal"],
#     title_param_dict = None,

#     label_parameters = ["N_spiral_turns"],
#     label_param_dict = None,

#     #sweep
#     sweep_dict = SWEEP_01_01,
#     only_iterations = None
# )

## y-axis (homogenity)

In [ ]:
cdp.yaxis_plot(
    translation_dict = TRANSLATE_PLOTLABELS,

    # theory
    magnetic_theory = None,
    temperature_theory = None,

    #paths
    output_folder = SWEEP_OUTPUT_FOLDER_1,
    input_folder = INPUT_FOLDER_1,
    modelfolder = FOLDER_01_XX,
    groupname = SWEEPNAME_01_01,
    modelnames = list(SWEEP_01_01.keys()),
    
    ending = HOMOGENEITY_END,
    data_export = SWEEP_EXPORT,

    #plotting
    fraction = 1.0,
    y_axis_params=Y_AXIS_PARAMS,

    plot_z = Z_VALUES[1:-1],
    insulator = False,
    
    width_limit_param = None,
    width_limit = None,

    title_parameters = ["I_spiral_terminal"],
    title_param_dict = None,

    label_parameters = ["N_spiral_turns"],
    label_param_dict = None,
    
    #sweep
    sweep_dict = SWEEP_01_01,
    only_iterations = None
)

## x-axis (longitudinal)

In [ ]:
cdp.xaxis_plot(
    translation_dict = TRANSLATE_PLOTLABELS,

    # theory
    magnetic_theory = None,
    temperature_theory = None,

    #paths
    output_folder = SWEEP_OUTPUT_FOLDER_1,
    input_folder = INPUT_FOLDER_1,
    modelfolder = FOLDER_01_XX,
    groupname = SWEEPNAME_01_01,
    modelnames = list(SWEEP_01_01.keys()),
    
    ending = LONGITUDINAL_END,
    data_export = SWEEP_EXPORT,

    #plotting
    fraction = 1.0,
    y_axis_params=Y_AXIS_PARAMS,

    plot_z = Z_VALUES[1:-1],
    insulator = False,

    length_limit_param = None,
    length_limit = None,

    title_parameters = ["I_spiral_terminal"],
    title_param_dict = None,

    label_parameters = ["N_spiral_turns"],
    label_param_dict = None,
    
    #sweep
    sweep_dict = SWEEP_01_01,
    only_iterations = None
)

# 01_03-Example combination of coil and grid design

## z-axis (depth)

In [ ]:
cdp.zaxis_plot(
    translation_dict = TRANSLATE_PLOTLABELS,

    # theory
    magnetic_theory = None,
    temperature_theory = None,

    #paths
    output_folder = SWEEP_OUTPUT_FOLDER_2,
    input_folder = INPUT_FOLDER_2,
    modelfolder = FOLDER_01_XX,
    groupname = SWEEPNAME_01_03,
    modelnames =list(SWEEP_01_03.keys()),
    
    ending = DEPTH_END,
    data_export = SWEEP_EXPORT,

    #plotting
    fraction = 1.0,
    y_axis_params=Y_AXIS_PARAMS,

    title_parameters = ["I_spiral_terminal", "N_spiral_turns", "N_grid"],
    title_param_dict = None,

    label_parameters = ["V_Grid"], 
    label_param_dict = SWEEP_PARAMETER_DICT_01_03,
    
    #sweep
    sweep_dict = SWEEP_01_03,
    only_iterations = None
)

## y-axis (homogenity)

In [ ]:
cdp.yaxis_plot(
    translation_dict = TRANSLATE_PLOTLABELS,

    # theory
    magnetic_theory = None,
    temperature_theory = None,

    #paths
    output_folder = SWEEP_OUTPUT_FOLDER_2,
    input_folder = INPUT_FOLDER_2,
    modelfolder = FOLDER_01_XX,
    groupname = SWEEPNAME_01_03,
    modelnames =list(SWEEP_01_03.keys()),
    
    ending = HOMOGENEITY_END,
    data_export = SWEEP_EXPORT,

    #plotting
    fraction = 1.0,
    y_axis_params=Y_AXIS_PARAMS,

    plot_z = Z_VALUES[1:-1],
    insulator = False,

    width_limit_param = None,
    width_limit = None,

    title_parameters = ["I_spiral_terminal", "N_spiral_turns", "N_grid"],
    title_param_dict = None,

    label_parameters = ["V_Grid"], 
    label_param_dict = SWEEP_PARAMETER_DICT_01_03,
    
    #sweep
    sweep_dict = SWEEP_01_03,
    only_iterations = None
)

## x-axis (longitudinal)

In [ ]:

cdp.xaxis_plot(
    translation_dict = TRANSLATE_PLOTLABELS,

    # theory
    magnetic_theory = None,
    temperature_theory = None,

    #paths
    output_folder = SWEEP_OUTPUT_FOLDER_2,
    input_folder = INPUT_FOLDER_2,
    modelfolder = FOLDER_01_XX,
    groupname = SWEEPNAME_01_03,
    modelnames =list(SWEEP_01_03.keys()),
    
    ending = LONGITUDINAL_END,
    data_export = SWEEP_EXPORT,

    #plotting
    fraction = 1.0,
    y_axis_params=Y_AXIS_PARAMS,

    plot_z = Z_VALUES[1:-1],
    insulator = False,

    length_limit_param = None,
    length_limit = None,

    title_parameters = ["I_spiral_terminal", "N_spiral_turns", "N_grid"],
    title_param_dict = None,

    label_parameters = ["V_Grid"], 
    label_param_dict = SWEEP_PARAMETER_DICT_01_03,
    
    #sweep
    sweep_dict = SWEEP_01_03,
    only_iterations = None
)

# 01_02-Number of grid (Sweep from Error Angle)

In [ ]:
only_iterations = [15] 
angles = list(range(0, 91, 3))

## y-axis (homogenity)

In [ ]:
cdp.yaxis_plot(
    translation_dict = TRANSLATE_PLOTLABELS,

    # theory
    magnetic_theory = None,
    temperature_theory = None,

    #paths
    output_folder = OUTPUT_FOLDER_1,
    input_folder = INPUT_FOLDER_1,
    modelfolder = FOLDER_01_XX,
    groupname = SWEEPNAME_01_02_LOL,
    modelnames = MODELS_01_02,
    
    ending = HOMOGENEITY_END,
    data_export = SWEEP_EXPORT,

    #plotting
    fraction = 1.0,
    y_axis_params=Y_AXIS_PARAMS,

    title_parameters = ["V_Grid"],
    title_param_dict = {r"$\theta$ $[^\circ]$": angles[only_iterations[0]]}, 

    label_parameters = ["N_grid"],
    label_param_dict={
        "left_out_lines": [0, 1, 2],
        },

    plot_z = Z_VALUES[-1:],
    insulator = False,

    width_limit_param = None,
    width_limit = None,

    # alibi sweep
    sweep_dict = SWEEP_01_02,
    display_left_out_lines = False,
    only_iterations = only_iterations,
)

## x-axis (longitudinal)

In [ ]:
cdp.xaxis_plot(
    translation_dict = TRANSLATE_PLOTLABELS,

    # theory
    magnetic_theory = None,
    temperature_theory = None,

    #paths
    output_folder = OUTPUT_FOLDER_1,
    input_folder = INPUT_FOLDER_1,
    modelfolder = FOLDER_01_XX,
    groupname = SWEEPNAME_01_02_LOL,
    modelnames = MODELS_01_02,
    
    ending = LONGITUDINAL_END,
    data_export = SWEEP_EXPORT,

    #plotting
    fraction = 1.0,
    y_axis_params=Y_AXIS_PARAMS,

    title_parameters = ["V_Grid"],
    title_param_dict = {r"$\theta$ $[^\circ]$": angles[only_iterations[0]]}, 

    label_parameters = ["N_grid"],
    label_param_dict={
        "left_out_lines": [0, 1, 2],
        },

    plot_z = Z_VALUES[:1],
    insulator = False,

    length_limit_param = None,
    length_limit = None,

    # alibi sweep
    sweep_dict = SWEEP_01_02,
    display_left_out_lines = False,
    only_iterations = only_iterations,
)

## xy-plane

In [ ]:
cdp.xyplane_plot(
    translation_dict = TRANSLATE_PLOTLABELS,

    #paths
    output_folder = OUTPUT_FOLDER_1,
    input_folder = INPUT_FOLDER_1,
    modelfolder = FOLDER_01_XX,
    groupname = SWEEPNAME_01_02_LOL,
    modelnames = MODELS_01_02,
    
    ending = XY_PLANE_END,
    data_export = SWEEP_EXPORT,

    #plotting
    fraction = 1.0,
    z_axis_params=Y_AXIS_PARAMS + ["T (K)"],

    title_parameters = ["V_Grid", "N_grid"],
    title_param_dict = {r"$\theta$ $[^\circ]$": angles[only_iterations[0]]}, 

    sweep_parameters_dict={
        "left_out_lines": [0, 1, 2],
        },

    length_limit_param = "conductor_all_length",
    length_limit = None,

    width_limit_param = "conductor_all_length",
    width_limit = None,

    # alibi sweep
    sweep_dict = SWEEP_01_02,
    display_left_out_lines = False,
    only_iterations = only_iterations,
)

## conductor-plane

In [ ]:
cdp.xyplane_plot(
    translation_dict = TRANSLATE_PLOTLABELS,

    #paths
    output_folder = OUTPUT_FOLDER_1,
    input_folder = INPUT_FOLDER_1,
    modelfolder = FOLDER_01_XX,
    groupname = SWEEPNAME_01_02_LOL,
    modelnames = MODELS_01_02,
    
    ending = CONDUCTOR_PLANE_END,
    data_export = SWEEP_EXPORT,

    #plotting
    fraction = 1.0,
    z_axis_params=CON_PLANE_PARAMS,

    title_parameters = ["V_Grid", "N_grid"],
    title_param_dict = {r"$\theta$ $[^\circ]$": angles[only_iterations[0]]}, 

    sweep_parameters_dict={
        "left_out_lines": [0, 1, 2],
        },
        
    length_limit_param = "conductor_all_length",
    length_limit = None,

    width_limit_param = "conductor_all_length",
    width_limit = None,

    # alibi sweep
    sweep_dict = SWEEP_01_02,
    display_left_out_lines = False,
    only_iterations = only_iterations,
)

# 01_02-Angle Error Analysis on Grid Design

In [ ]:
if False:
    z_values = [-1e-05, -9.66667e-06, -9.33333e-06, -9e-06, -8.66667e-06, -8.33333e-06, -8e-06, -7.66667e-06, -7.33333e-06, -7e-06, -6.66667e-06, -6.33333e-06, -6e-06, -5.66667e-06, -5.33333e-06, -5e-06, -4.66667e-06, -4.33333e-06, -4e-06, -3.66667e-06, -3.33333e-06, -3e-06, -2.66667e-06, -2.33333e-06, -2e-06, -1.66667e-06, -1.33333e-06, -1e-06, -6.66667e-07, -3.33333e-07, 0.0]
    z_values = z_values[1:-1] # drop the first and last value
    z_values = [z for i,z in enumerate(z_values) if i % 2 == 0] # drop every second value
else:
    # z_values = [-1e-05, -8.20457e-06, -6.40914e-06, -4.61371e-06, -2.81827e-06, -1.14985e-06]
    z_values = [-9e-06, -8e-06, -7e-06, -6e-06, -5e-06, -4e-06, -3e-06, -2e-06, -1e-06]
    # z_values = Z_VALUES

print(z_values)


In [ ]:
df_stats = pd.DataFrame() 
for z in z_values:
    df_statistics_z = cdp.angle_error_plot(
        translation_dict = TRANSLATE_PLOTLABELS,

        #paths
        output_folder = SWEEP_OUTPUT_FOLDER_1,
        input_folder = INPUT_FOLDER_1,
        modelfolder = FOLDER_01_XX,
        groupname = SWEEPNAME_01_02,
        modelnames = list(SWEEP_01_02.keys()),
        ending = DEPTH_END,
        data_export = SWEEP_EXPORT,

        fraction = 1.0,
        title_parameters = None,
        title_param_dict = None, 

        label_parameters = ["N_grid"],
        label_param_dict={
            "left_out_lines": [0, 1, 2],
            r"Set angle (°)": list(range(0, 91, 3))
            },

        plot_z = [z],

        sweep_dict = SWEEP_01_02,

        display_statistics = True,
        plot_total_mean = False,
        )
    df_stats = pd.concat([df_stats, df_statistics_z], ignore_index=True)

## Angle Analysis Statistics

In [ ]:
display(df_stats)
df = df_stats.copy()

In [ ]:
max_error = df["Max Angle Error (°)"].max()
print(f"Max Angle Error (m°): {max_error*1e3:.3g}")

In [ ]:
# statistic propagation
def combine_stats(group):
    # Total Mean
    grand_mean = group["Mean Angle Error (°)"].mean()

    # Total Standard Deviation
    variance_within = (group["Std Dev (°)"] ** 2).mean()
    variance_between = group["Mean Angle Error (°)"].var(ddof=0)
    total_std = np.sqrt(variance_within + variance_between)

    return pd.Series(
        {
            "Total Mean (°)": grand_mean,
            "Total Std (°)": total_std,
        }
    )


df_per_z = df.groupby("z", group_keys=False).apply(combine_stats)
display(df_per_z)

df_per_lines = df.groupby("Left out lines", group_keys=False).apply(
    combine_stats
)
display(df_per_lines)

df_total = combine_stats(df).to_frame().T
display(df_total)

# End

In [ ]:
tl.log_message("Finished creating standard plots for all models.")

## Angle Sweep Table

In [ ]:
# import voltage_grid as vg

In [ ]:
# angles = list(range(0, 91, 3))  # angles from 0 to 90 degrees in steps of 3 degrees
# sweep_params_1_2, sweep_values_1_2 = vg.get_voltage_sweep_dict(
#     angles = angles,
#     magnitude = 10e-6,
#     conductor_grid_length = 60e-6,
# ) 

In [ ]:
# angles = list(range(0, 91, 3))
# sweep_params_1_2, sweep_values_1_2 = vg.get_voltage_sweep_dict(
#     angles=angles,
#     magnitude=10e-6,
#     conductor_grid_length=60e-6,
# )

# rows = []
# for angle, sweep_row in zip(angles, sweep_values_1_2):
#     row = {"angle": angle}
#     for param_name, value in zip(sweep_params_1_2, sweep_row):
#         row[param_name] = float(str(value).replace("[V]", ""))
#     rows.append(row)

# df = pd.DataFrame(rows)

In [ ]:
# with pd.option_context('display.max_rows', None, 'display.max_columns', None, 'display.width', 2000):
#     print(df.T)

In [ ]:
# import pandas as pd

# # 1. Sicherstellen, dass die Iterationsspalte 'idx' ganz vorne steht
# if 'idx' not in df.columns:
#     df.insert(0, 'idx', range(1, len(df) + 1))
# elif df.columns[0] != 'idx':
#     cols = ['idx'] + [col for col in df.columns if col != 'idx']
#     df = df[cols]

# # 2. Formatierungsfunktion für die wissenschaftliche Notation
# def format_latex_num(val):
#     if pd.isna(val):
#         return ""
#     return f"\\num{{{val:.2e}}}"

# # 3. Formatierer für die ersten beiden Spalten festlegen
# formatters = {
#     'idx': lambda x: f"\\num{{{int(x)}}}" if not pd.isna(x) else "",
#     'theta_voltage': lambda x: f"\\num{{{int(x)}}}" if not pd.isna(x) else ""
# }

# # Füge alle Terminal-Spalten (V01 bis V20) hinzu
# for col in df.columns:
#     if col.startswith('V'):
#         formatters[col] = format_latex_num

# # 4. Platzhalter-Header ohne geschweifte Klammern definieren
# placeholder_headers = []
# for col in df.columns:
#     if col == 'idx':
#         placeholder_headers.append("HEADER_IDX")
#     elif col == 'theta_voltage':
#         placeholder_headers.append("HEADER_THETA")
#     else:
#         placeholder_headers.append(f"HEADER_V_{col[1:]}")

# # 5. Den LaTeX-Code generieren
# latex_code = df.to_latex(
#     index=False,
#     header=placeholder_headers,
#     formatters=formatters,
#     column_format=f"ccr*{{{len(df.columns)-2}}}{{r}}",
#     escape=False,
#     position='h'
# )

# # 6. Platzhalter im fertigen String durch reines LaTeX-Symbol (OHNE Einheiten) ersetzen
# latex_code = latex_code.replace("HEADER_IDX", r"\(idx\)")
# latex_code = latex_code.replace("HEADER_THETA", r"\(\theta_{\mathrm{voltage}}\)")

# for col in df.columns:
#     if col.startswith('V'):
#         num = col[1:]
#         placeholder = f"HEADER_V_{num}"
#         latex_target = f"\\(V_{{\\mathrm{{{num}}}}}\\)"
#         latex_code = latex_code.replace(placeholder, latex_target)

# # 7. Ausgabe des fertigen Codes
# print(latex_code)
